# Train skin-conditions model (5 labels)

This notebook trains a multi-label classifier over: `redness`, `acne`, `pigmentation`, `wrinkles`, `normal_skin`.

Expected dataset layout:
- Repo root: `data/`
- One folder per class or combo folder (e.g. `acne/`, `acne_wrinkles/`, `normal_skin/`, etc.)
- If a folder contains an explicit split layout (Roboflow/YOLO style), training uses only `train/` and evaluation uses `valid/` (fallback `test/`).

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import tensorflow as tf

keras = tf.keras

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

LABELS = ["redness", "acne", "pigmentation", "wrinkles", "normal_skin"]
_WRINKLES_IDX = LABELS.index("wrinkles")

@dataclass(frozen=True)
class Config:
    data_root: Path = Path("data") if Path("data").is_dir() else Path("../data")
    out_model: Path = Path("resnet50_5labels.keras")
    img_size: tuple[int, int] = (224, 224)
    batch_size: int = 32
    epochs: int = 8
    lr: float = 3e-4
    val_frac: float = 0.2
    seed: int = 1337

CFG = Config()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def labels_from_combo_folder(folder_name, labels):
    toks = folder_name.lower().replace("-", "_").split("_")
    y = np.zeros((len(labels),), dtype=np.float32)
    for i, lab in enumerate(labels):
        parts = lab.lower().replace("-", "_").split("_")
        if all(part in toks for part in parts):
            y[i] = 1.0
    return y


In [ ]:
def list_samples(data_root, labels):
    samples = []
    if not data_root.exists():
        return samples
    for combo_dir in sorted([p for p in data_root.iterdir() if p.is_dir()], key=lambda p: p.name.lower()):
        y = labels_from_combo_folder(combo_dir.name, labels)
        if float(np.sum(y)) == 0.0:
            continue
        for p in combo_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                samples.append((str(p), y))
    return samples

def _iter_image_files(root):
    if not root.exists() or not root.is_dir():
        return []
    out = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            out.append(str(p))
    return out

def _find_explicit_split_root(combo_dir):
    def looks_like_split_root(p):
        train_dir = p / "train"
        if not train_dir.is_dir():
            return False
        if (train_dir / "images").is_dir():
            return True
        return any(Path(x).suffix.lower() in IMG_EXTS for x in _iter_image_files(train_dir)[:1])

    if looks_like_split_root(combo_dir):
        return combo_dir

    children = [p for p in combo_dir.iterdir() if p.is_dir()]
    if len(children) == 1 and looks_like_split_root(children[0]):
        return children[0]

    for ch in children:
        if looks_like_split_root(ch):
            return ch
    return None

def _split_dir_images(split_dir):
    img_root = split_dir / "images"
    if img_root.is_dir():
        return _iter_image_files(img_root)
    return _iter_image_files(split_dir)

def split_samples(samples, *, val_frac, seed):
    rng = random.Random(seed)
    rng.shuffle(samples)

    val_n = int(len(samples) * val_frac)
    val_samples = samples[:val_n]
    train_samples = samples[val_n:]
    return train_samples, val_samples

def build_train_val_samples(
    data_root,
    labels,
    *,
    val_frac,
    seed,
    prefer_eval_split="valid",
):
    explicit_train = []
    explicit_val = []
    unsplit = []

    if not data_root.exists():
        return [], []

    combo_dirs = [p for p in data_root.iterdir() if p.is_dir()]
    for combo_dir in sorted(combo_dirs, key=lambda p: p.name.lower()):
        y = labels_from_combo_folder(combo_dir.name, labels)
        if float(np.sum(y)) == 0.0:
            continue

        split_root = _find_explicit_split_root(combo_dir)
        if split_root is None:
            for p in combo_dir.rglob("*"):
                if p.is_file() and p.suffix.lower() in IMG_EXTS:
                    unsplit.append((str(p), y))
            continue

        train_paths = _split_dir_images(split_root / "train")

        eval_dir = None
        if prefer_eval_split == "valid" and (split_root / "valid").is_dir():
            eval_dir = split_root / "valid"
        elif prefer_eval_split == "test" and (split_root / "test").is_dir():
            eval_dir = split_root / "test"
        elif (split_root / "valid").is_dir():
            eval_dir = split_root / "valid"
        elif (split_root / "val").is_dir():
            eval_dir = split_root / "val"
        elif (split_root / "test").is_dir():
            eval_dir = split_root / "test"

        eval_paths = _split_dir_images(eval_dir) if eval_dir is not None else []

        explicit_train.extend([(p, y) for p in train_paths])
        explicit_val.extend([(p, y) for p in eval_paths])

        print(f"explicit split detected: {combo_dir.name} -> {split_root.name}  train={len(train_paths)}  eval={len(eval_paths)}")

    rand_train, rand_val = split_samples(
        unsplit,
        val_frac=val_frac,
        seed=seed,
    )

    train_samples = explicit_train + rand_train
    val_samples = explicit_val + rand_val

    return train_samples, val_samples

In [3]:
def _build_default_augmenter() -> keras.Model:
    return keras.Sequential(
        [
            keras.layers.RandomFlip("horizontal"),
            keras.layers.RandomRotation(0.05),
            keras.layers.RandomZoom(0.10),
            keras.layers.RandomContrast(0.10),
        ],
        name="augment_default",
    )

def _build_gentle_augmenter() -> keras.Model:
    return keras.Sequential(
        [
            keras.layers.RandomFlip("horizontal"),
            keras.layers.RandomRotation(0.05),
            keras.layers.RandomZoom(0.10),
        ],
        name="augment_gentle",
    )

def _decode_and_resize(path: tf.Tensor, y: tf.Tensor, *, img_size: tuple[int, int]) -> tuple[tf.Tensor, tf.Tensor]:
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, img_size, method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32) / 255.0
    return img, y

def build_dataset(
    samples: list[tuple[str, np.ndarray]],
    *,
    img_size: tuple[int, int],
    batch_size: int,
    training: bool,
    seed: int,
) -> tf.data.Dataset:
    paths = np.array([p for (p, _) in samples], dtype=np.str_)
    ys = np.stack([y for (_, y) in samples]).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    if training:
        ds = ds.shuffle(buffer_size=min(len(samples), 4096), seed=seed, reshuffle_each_iteration=True)

    ds = ds.map(lambda p, y: _decode_and_resize(p, y, img_size=img_size), num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        default_aug = _build_default_augmenter()
        gentle_aug = _build_gentle_augmenter()

        def _maybe_augment(x: tf.Tensor, y: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
            is_wrinkles = y[_WRINKLES_IDX] > 0.5

            def _do_gentle() -> tf.Tensor:
                return gentle_aug(x, training=True)

            def _do_default() -> tf.Tensor:
                return default_aug(x, training=True)

            x_aug = tf.cond(is_wrinkles, _do_gentle, _do_default)
            return x_aug, y

        ds = ds.map(_maybe_augment, num_parallel_calls=tf.data.AUTOTUNE)

    preprocess = tf.keras.applications.resnet50.preprocess_input
    ds = ds.map(lambda x, y: (preprocess(x * 255.0), y), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [4]:
def build_model(*, num_labels: int, img_size: tuple[int, int]) -> keras.Model:
    inputs = keras.Input(shape=(img_size[0], img_size[1], 3), name="image")

    base = keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling="avg",
    )
    base.trainable = False

    x = keras.layers.Dropout(0.2)(base.output)
    outputs = keras.layers.Dense(num_labels, activation="sigmoid", name="probs")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name="resnet50_multilabel")
    return model

In [ ]:
seed_everything(CFG.seed)

samples = list_samples(CFG.data_root, LABELS)

print("total images (all sources):", len(samples))

train_samples, val_samples = build_train_val_samples(
    CFG.data_root,
    LABELS,
    val_frac=CFG.val_frac,
    seed=CFG.seed,
    prefer_eval_split="valid",
)

print("train:", len(train_samples), "val:", len(val_samples))

train_ds = build_dataset(
    train_samples,
    img_size=CFG.img_size,
    batch_size=CFG.batch_size,
    training=True,
    seed=CFG.seed,
)
val_ds = build_dataset(
    val_samples,
    img_size=CFG.img_size,
    batch_size=CFG.batch_size,
    training=False,
    seed=CFG.seed,
)

model = build_model(num_labels=len(LABELS), img_size=CFG.img_size)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CFG.lr),
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[
        keras.metrics.BinaryAccuracy(name="bin_acc", threshold=0.5),
        keras.metrics.AUC(name="auc", multi_label=True, num_labels=len(LABELS)),
    ],
)
model.summary()

ckpt_cb = keras.callbacks.ModelCheckpoint(
    filepath=str(CFG.out_model),
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CFG.epochs,
    callbacks=[ckpt_cb],
)

last_val = float(history.history.get("val_loss", [float("nan")])[-1])
print("done. last_val_loss=", last_val)
print("saved best model to:", CFG.out_model)
print("labels:", LABELS)

TensorFlow: 2.20.0
GPUs: 0
data_root: E:\AISkincareRecommendationCognitiveWorks\data
total images (all sources): 2499
explicit split detected: acne -> data-2  train=823  eval=56
train: 2086 val: 365


Model: "resnet50_multilabel"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ image[0][0]       │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/8
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 1000ms/step - auc: 0.6447 - bin_acc: 0.7199 - loss: 0.5662
Epoch 1: val_loss improved from None to 0.38787, saving model to resnet50_5labels.keras

Epoch 1: finished saving model to resnet50_5labels.keras
66/66 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - auc: 0.7330 - bin_acc: 0.8035 - loss: 0.4444 - val_auc: 0.8200 - val_bin_acc: 0.8444 - val_loss: 0.3879
Epoch 2/8
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8737 - bin_acc: 0.8812 - loss: 0.2992
Epoch 2: val_loss improved from 0.38787 to 0.31162, saving model to resnet50_5labels.keras

Epoch 2: finished saving model to resnet50_5labels.keras
66/66 ━━━━━━━━━━━━━━━━━━━━ 89s 1s/step - auc: 0.8884 - bin_acc: 0.8919 - loss: 0.2772 - val_auc: 0.8957 - val_bin_acc: 0.8690 - val_loss: 0.3116
Epoch 3/8
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 949ms/step - auc: 0.9259 - bin_acc: 0.9096 - loss: 0.2377
Epoch 3: val_loss improved from 0.31162 to 0.26837, saving model to resnet50_5labels.keras

Epoch 3: finished saving model 